# Getting Started with Monitoring and Observability

Welcome to the Firewall Automation Hackathon! This notebook will guide you through implementing monitoring and observability for the Firewall Automation agents.

## Overview

Monitoring and observability are critical for understanding agent performance, debugging issues, and ensuring system reliability. This implementation focuses on:
- AWS Bedrock AgentCore Monitoring and Observability
- CloudWatch Logs and Metrics
- AWS X-Ray for distributed tracing

## Related Jira Task

- [FWAUTO-20: Implement monitoring and observability (Phase 1)](https://your-jira-instance.atlassian.net/browse/FWAUTO-20)

## Key Metrics to Track

### Agent Metrics
- **Invocation count**: Number of times each agent is called
- **Success rate**: Percentage of successful agent invocations
- **Response time**: Average, P95, P99 latency
- **Error rate**: Failures by agent type and error category
- **Tool usage**: Which tools are used most frequently

### Integration Health
- **OpenSearch query latency**: Firewall log query performance
- **DynamoDB access patterns**: Account details lookup performance
- **ServiceNow API availability**: SNOW integration uptime
- **Azure DevOps API response times**: Git operations performance

### Business Metrics
- **Active users**: Unique users per day/week
- **Firewall rules created**: Number of successful rule changes
- **Pull requests submitted**: Automation effectiveness
- **User satisfaction**: Feedback and error resolution time

## AWS Observability Documentation

For comprehensive guidance on implementing observability in AgentCore, refer to the official documentation:
- [AWS Bedrock AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

## Step 1: Copy the Base Agent Code

Copy the supervisor agent code to your workspace so you can modify it.

In [ ]:
%%bash
# Copy the base agent code from the repository
cp -r /home/sagemaker-user/IST-AWS-Firewall-Automation/agent .

# Remove the existing bedrock_agentcore.yaml configuration
rm -f agent/.bedrock_agentcore.yaml

In [ ]:
# View the base agent python code
with open('agent/src/agent.py', 'r') as f:
    print(f.read())

## Step 2: Review the README

Study the README.md file in this folder to understand the monitoring requirements and metrics to track.

In [ ]:
# View the README for implementation guidance
with open('README.md', 'r') as f:
    print(f.read())

## Step 3: Configure Observability Settings

Configure monitoring and observability for your agent deployment. This involves:

### CloudWatch Logs
- Agent execution logs
- Error tracking and alerting
- Log aggregation across all agents

### CloudWatch Metrics
- Custom metrics for agent performance
- Dashboard creation for visualization
- Alarms for error thresholds

### AWS X-Ray
- Distributed tracing across agents
- Service map visualization
- Performance bottleneck identification

### AgentCore Observability
The deployment step (Step 5) will configure AgentCore's built-in observability features. Review the [AWS Bedrock AgentCore Observability documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) to understand:
- How to enable observability in your agent runtime
- What metrics are automatically collected
- How to configure CloudWatch integration
- How to set up X-Ray tracing

In [ ]:
# Review AWS Observability Documentation
# No code changes needed in agent.py for basic observability
# Observability is configured during deployment (see Step 5)

print("Review the AWS Bedrock AgentCore Observability documentation:")
print("https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html")
print("\nKey topics to understand:")
print("- Enabling observability in agent runtime configuration")
print("- CloudWatch Logs integration")
print("- CloudWatch Metrics for agent performance")
print("- AWS X-Ray distributed tracing")
print("- Custom metrics and alarms")

## Step 4: Plan Your Observability Strategy

Before deploying with observability enabled, plan your monitoring strategy.

In [ ]:
# Plan your observability configuration
# Consider the following:

print("Observability Configuration Checklist:")
print("\n1. CloudWatch Log Groups:")
print("   - Log retention period (7 days, 30 days, etc.)")
print("   - Log group naming convention")
print("   - Log stream organization")
print("\n2. CloudWatch Metrics:")
print("   - Which custom metrics to track")
print("   - Metric namespaces and dimensions")
print("   - Dashboard layout and widgets")
print("\n3. CloudWatch Alarms:")
print("   - Error rate thresholds")
print("   - Latency P99 thresholds")
print("   - Integration failure alerts")
print("   - SNS topic for notifications")
print("\n4. AWS X-Ray:")
print("   - Sampling rate (1%, 10%, 100%)")
print("   - Trace retention period")
print("   - Service map configuration")
print("\n5. AgentCore Observability:")
print("   - Enable in runtime configuration (Step 5)")
print("   - Verify automatic metric collection")
print("   - Configure trace export to X-Ray")

## Step 5: Deploy with Observability Enabled

Deploy your agent with observability and monitoring configuration. This is where you enable AgentCore's observability features, CloudWatch integration, and X-Ray tracing.

Refer to the [AWS Bedrock AgentCore Observability documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) for specific configuration parameters.

In [ ]:
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# TODO: Set your agent name
agent_name = <AGENT_NAME>  # e.g., "firewall-supervisor-agent"

# Initialize the runtime toolkit
region = "ap-southeast-2"

agentcore_runtime = Runtime()

# Configure the deployment WITH OBSERVABILITY ENABLED
# Review https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html
# for additional observability configuration parameters

response = agentcore_runtime.configure(
    agent_name=agent_name,
    entrypoint=<ENTRYPOINT>,  # TODO: Set your entrypoint file, e.g., "agent/src/agent.py"
    execution_role="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    code_build_execution_role="arn:aws:iam::123456789012:role/YourCodeBuildRole",
    auto_create_ecr=True,
    requirements_file=<REQUIREMENTS_FILE>,  # TODO: Set your requirements file, e.g., "agent/src/requirements.txt"
    region=region,
    memory_mode="STM_ONLY",
    # TODO: Add observability configuration here
    # Refer to AWS documentation for parameters like:
    # - CloudWatch log group configuration
    # - X-Ray tracing settings
    # - Custom metrics configuration
)

print("Configuration completed:", response)

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    runtime_id = status_response.agent["agentRuntimeId"]

    # Update the runtime to be deployed in VPC with observability
    client = boto3.client("bedrock-agentcore-control", region_name=region)

    response = client.update_agent_runtime(
        agentRuntimeId=runtime_id,
        networkConfiguration={
            "networkMode": "VPC",
            "networkModeConfig": {
                "subnets": ["subnet-xxxxxxxxxxxxxxxxx", "subnet-yyyyyyyyyyyyyyyyy"],
                "securityGroups": ["sg-xxxxxxxxxxxxxxxxx"],
            },
        },
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": f"123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-{agent_name}"
            }
        },
        roleArn="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
        # TODO: Add observability settings here during update_agent_runtime
        # Configure CloudWatch Logs, Metrics, and X-Ray tracing
    )
    
    print(f"Agent deployed successfully with observability enabled!")
    print(f"CloudWatch Logs: Check log group for agent execution logs")
    print(f"CloudWatch Metrics: View custom metrics in CloudWatch dashboard")
    print(f"X-Ray: View distributed traces in AWS X-Ray console")

## Step 6: Verify Observability

After deployment, verify that monitoring and observability are working correctly.

## Step 7: Create CloudWatch Dashboards

Create CloudWatch dashboards to visualize agent performance and system health.

## Tips and Best Practices

### CloudWatch Logs
- **Log retention**: Set appropriate retention periods (7-30 days for development, longer for production)
- **Log organization**: Use consistent naming conventions for log groups and streams
- **Structured logging**: Use JSON format for easier querying and analysis
- **Error tracking**: Tag errors with severity levels and error codes

### CloudWatch Metrics
- **Custom metrics**: Track agent-specific metrics beyond default AgentCore metrics
- **Metric dimensions**: Use dimensions like agent_name, environment, version for filtering
- **Dashboard design**: Create role-specific dashboards (engineers, operations, business)
- **Metric aggregation**: Use statistics like Average, Sum, Maximum for different metric types

### CloudWatch Alarms
- **Threshold tuning**: Start conservative, adjust based on actual usage patterns
- **Alarm actions**: Configure SNS topics for notifications to Slack/email
- **Composite alarms**: Combine multiple metrics for sophisticated alerting
- **Alarm testing**: Test alarm triggers before production deployment

### AWS X-Ray
- **Sampling strategy**: Use 100% sampling during development, 1-10% in production
- **Trace retention**: Balance cost with debugging needs (typically 7-30 days)
- **Service map**: Review service dependencies and identify bottlenecks
- **Annotations**: Add custom annotations to traces for better filtering

### Performance Optimization
- **Baseline metrics**: Establish performance baselines early
- **Latency tracking**: Monitor P50, P95, P99 latencies, not just averages
- **Error rate monitoring**: Set error rate thresholds and investigate spikes
- **Resource utilization**: Track memory, CPU usage to optimize runtime configuration

### Security and Compliance
- **Audit logging**: Log all agent invocations with user context
- **PII redaction**: Ensure logs don't contain sensitive information
- **Access control**: Use IAM policies to restrict CloudWatch access
- **Compliance**: Retain logs according to organizational requirements

## Resources

- README: `README.md` in this folder
- Jira: [FWAUTO-20](https://your-jira-instance.atlassian.net/browse/FWAUTO-20)
- AWS Bedrock AgentCore Observability: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html
- CloudWatch Documentation: https://docs.aws.amazon.com/cloudwatch/
- AWS X-Ray Documentation: https://docs.aws.amazon.com/xray/
- `./06-AgentCore-observability` for monitoring-specific examples
- `/home/sagemaker-user/amazon-bedrock-agentcore-samples/` for more examples from AWS